In [ ]:
import os
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item

In [ ]:
LITE_MODE=True

load_dotenv(override=True)
HF_token=os.getenv('HF_token')
login(HF_token)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [ ]:
openai = OpenAI()

# Fine tune the data
fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [ ]:
len(fine_tune_validation)

Step 1 prepare data to jsonl files format to upload to OpenAI

In [ ]:
def messsages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [ ]:
messsages_for(fine_tune_validation[0])

In [ ]:
def make_jsonl(items):
    result=""
    for item in items:
        messages = messsages_for(item)
        message_str = json.dumps(messages)
        result+='{"messages":'+message_str+'}\n' # {messages: [ {"role": "user", "content":},{"role": "assistant", "content":}
    return result.strip()

In [ ]:
def write_jsonl(items,filename):
    with open(filename,'w') as f:
        jsonl=make_jsonl(items)
        f.write(jsonl)
write_jsonl(fine_tune_train,'jsonl/fine_tune_train.jsonl')

In [ ]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [ ]:
# Now we need to upload thefiles
with open("jsonl/fine_tune_train.jsonl",'rb') as f:
    train_file = openai.files.create(file=f,purpose="fine-tune")

In [ ]:
train_file

In [ ]:
with open("jsonl/fine_tune_validation.jsonl",'rb') as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune") 

Step 2 And now to fine tune

In [ ]:
openai.fine_tuning.jobs.create(
    training_file = train_file.id,
    validation_file = validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

In [ ]:
openai.fine_tuning.jobs.list(limit=1)


In [ ]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [ ]:
openai.fine_tuning.jobs.retrieve(job_id).status

In [ ]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

STEP 3
Fine tuned model

In [ ]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [ ]:
fine_tuned_model_name

In [ ]:
def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role":"user","content":message}
    ]

In [ ]:
test_messages_for(test[0])

In [ ]:
# inference function

def gpt_4_1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages= test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content


In [ ]:
print(test[0].price)